# Model Train

In [ ]:
import torch

from model import Monster
from train import MyTrain
from evaluate import MyEvaluate
from preprocessing import apply_augmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Monster(
    vocab_size=len(vocab),
    num_classes=2,
    embedding_dim=128,
    padding_idx=0,
    chunk_size=256,
    chunk_stride=224,
    token_hidden=128,
    token_layers=1,
    sent_hidden=128,
    sent_layers=1,
    dropout=0.2,
    meta_dim=0,                
    meta_out=64,
    use_domain_embedding=True,
    num_domains=3,
    domain_emb_dim=32,
).to(device)


# =========================================================
# 4) trainer
#    하이퍼파라미터는 여기서 바로 바꾸면 됨
# =========================================================
trainer = MyTrain(
    model=model,
    device=device,
    lr=1e-3,
    weight_decay=1e-2,
    max_grad_norm=5.0,
    epochs=20,
    patience=5,
    scheduler_patience=2,
    scheduler_factor=0.5,
    class_weights=None,            # 필요하면 [w0, w1]
    augmentation_fn=apply_augmentation,
    use_augmentation=True,
    save_path=None,                # 저장 안 할 거면 None
)


# =========================================================
# 5) train
# =========================================================
history = trainer.fit(train_loader, val_loader)


# =========================================================
# 6) evaluate
# =========================================================
evaluator = MyEvaluate(model=model, device=device)
result = evaluator.evaluate(val_loader)
evaluator.print_summary(result)